# Exercise 3: Conversational Memory

**Level:** Basic

LLMs are stateless — each call is independent. To build chatbots and agents that remember context, you need **memory**. In this exercise, you will learn how LangChain manages conversation history.

**What you will learn:**
- Why memory matters for agents
- ChatMessageHistory for storing messages
- RunnableWithMessageHistory for automatic context injection
- Comparing behavior with and without memory
- Trimming messages to manage context length
## 1. Setup & Installation

In [ ]:
!pip install langchain langchain-openai langchain-community -q
import os
os.environ["OPENAI_API_KEY"] = "your-key-here"

## 2. The Problem: Stateless LLMs

Let's see what happens when you chat with a model that has no memory.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful travel assistant."),
    ("human", "{input}")
])

chain = prompt | model | StrOutputParser()

# Turn 1
response1 = chain.invoke({"input": "I'm planning a trip to Istanbul."})
print(f"Turn 1: {response1}")

print("\n" + "="*60 + "\n")

# Turn 2 — the model has NO IDEA what "there" refers to
response2 = chain.invoke({"input": "What's the best time to visit there?"})
print(f"Turn 2: {response2}")

Notice in Turn 2, the model does not know what "there" refers to. It lost all context from Turn 1. This is the fundamental problem memory solves.

## 3. ChatMessageHistory — The Message Store

LangChain provides `ChatMessageHistory` as a simple in-memory message store.

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# Create an in-memory message store
history = InMemoryChatMessageHistory()

# Add messages manually
history.add_message(HumanMessage(content="I'm planning a trip to Istanbul."))
history.add_message(AIMessage(content="Istanbul is a wonderful choice! When are you planning to go?"))
history.add_message(HumanMessage(content="Sometime in spring."))
history.add_message(AIMessage(content="April and May are ideal for Istanbul. The weather is pleasant."))

# Inspect the history
print(f"Message count: {len(history.messages)}")
print()
for msg in history.messages:
    print(f"[{msg.type}]: {msg.content}")

## 4. Prompt with Message History Placeholder

To use memory, your prompt template needs a `MessagesPlaceholder` where the history will be injected.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Create a prompt with a history placeholder
prompt_with_history = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful travel assistant. Be concise."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

# See what it looks like with history injected
formatted = prompt_with_history.invoke({
    "history": history.messages,
    "input": "What about hotels?"
})

print("Full prompt with history:")
for msg in formatted.messages:
    print(f"  [{msg.type}]: {msg.content[:80]}..." if len(msg.content) > 80 else f"  [{msg.type}]: {msg.content}")

## 5. RunnableWithMessageHistory — Automatic Memory Management

`RunnableWithMessageHistory` wraps any chain and automatically:
1. Loads history before each call
2. Appends the user input and AI response after each call

It uses **session IDs** to keep separate conversations separate.

In [ ]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful travel assistant. Keep answers to 2-3 sentences."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

chain = prompt | model | parser

# Store for different sessions
session_store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    """Get or create a message history for a session."""
    if session_id not in session_store:
        session_store[session_id] = InMemoryChatMessageHistory()
    return session_store[session_id]

# Wrap the chain with message history
chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

print("Chain with memory is ready!")

In [ ]:
# Turn 1
config = {"configurable": {"session_id": "user-123"}}

response1 = chain_with_memory.invoke(
    {"input": "I'm planning a trip to Istanbul."},
    config=config
)
print(f"Turn 1: {response1}")

In [ ]:
# Turn 2 — NOW the model knows "there" = Istanbul!
response2 = chain_with_memory.invoke(
    {"input": "What's the best time to visit there?"},
    config=config
)
print(f"Turn 2: {response2}")

# Turn 3 — Context continues to accumulate
response3 = chain_with_memory.invoke(
    {"input": "And what about food? What should I try?"},
    config=config
)
print(f"Turn 3: {response3}")

# Let's see what the history looks like now
history = get_session_history("user-123")
print(f"Total messages in history: {len(history.messages)}\n")
for msg in history.messages:
    preview = msg.content[:100] + "..." if len(msg.content) > 100 else msg.content
    print(f"[{msg.type}]: {preview}")

## 6. Multiple Sessions — Isolated Conversations

Different session IDs maintain completely separate conversation histories.

In [ ]:
# User A talks about Istanbul
config_a = {"configurable": {"session_id": "user-A"}}
resp = chain_with_memory.invoke({"input": "Tell me about Istanbul."}, config=config_a)
print(f"User A: {resp}\n")

# User B talks about Tokyo
config_b = {"configurable": {"session_id": "user-B"}}
resp = chain_with_memory.invoke({"input": "Tell me about Tokyo."}, config=config_b)
print(f"User B: {resp}\n")

# User A continues — should reference Istanbul, not Tokyo
resp = chain_with_memory.invoke({"input": "How do I get there from London?"}, config=config_a)
print(f"User A follow-up: {resp}\n")

# User B continues — should reference Tokyo, not Istanbul
resp = chain_with_memory.invoke({"input": "How do I get there from London?"}, config=config_b)
print(f"User B follow-up: {resp}")

## 7. Comparing With and Without Memory

Let's run the exact same conversation with both chains side by side to really see the difference.

In [ ]:
# Chain WITHOUT memory (from Section 2)
chain_no_memory = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a helpful travel assistant. Keep answers to 1 sentence."),
        ("human", "{input}")
    ])
    | model
    | parser
)

conversation = [
    "My name is Alex and I love Italian food.",
    "What city should I visit based on my preferences?",
    "What's my name?",
]

config_compare = {"configurable": {"session_id": "compare-session"}}

print("=" * 60)
print("WITH MEMORY vs WITHOUT MEMORY")
print("=" * 60)

for turn in conversation:
    print(f"\nHuman: {turn}")
    
    with_mem = chain_with_memory.invoke({"input": turn}, config=config_compare)
    without_mem = chain_no_memory.invoke({"input": turn})
    
    print(f"  With memory:    {with_mem}")
    print(f"  Without memory: {without_mem}")

## 8. Trimming Messages — Managing Context Length

As conversations grow, they can exceed the model's context window. **Message trimming** keeps the conversation manageable while preserving important context.

In [ ]:
from langchain_core.messages import trim_messages, HumanMessage, AIMessage, SystemMessage

# Simulate a long conversation
long_history = [
    SystemMessage(content="You are a travel assistant."),
    HumanMessage(content="I want to visit Europe."),
    AIMessage(content="Europe is great! What countries interest you?"),
    HumanMessage(content="Maybe Italy or Spain."),
    AIMessage(content="Both are wonderful. Italy has Rome and Florence."),
    HumanMessage(content="I love art and history."),
    AIMessage(content="Then Florence is perfect for you!"),
    HumanMessage(content="How long should I stay?"),
    AIMessage(content="I recommend 4-5 days for Florence."),
    HumanMessage(content="What about food?"),
    AIMessage(content="Try bistecca alla fiorentina and ribollita."),
    HumanMessage(content="Book me a hotel near the Uffizi."),
]

print(f"Total messages: {len(long_history)}")
print(f"Total characters: {sum(len(m.content) for m in long_history)}")

# Trim to keep only the most recent messages
trimmed = trim_messages(
    long_history,
    max_tokens=200,
    strategy="last",          # Keep the LAST messages
    token_counter=len,         # Simple character-based counting (use model for token counting)
    include_system=True,       # Always keep the system message
    allow_partial=False,       # Don't split messages
    start_on="human",          # Start trimmed history on a human message
)

print(f"\nAfter trimming:")
print(f"Remaining messages: {len(trimmed)}")
for msg in trimmed:
    print(f"  [{msg.type}]: {msg.content}")

## 9. Integrating Trimming into Your Chain

You can add trimming directly into your LCEL chain so it happens automatically.

In [ ]:
from langchain_core.messages import trim_messages
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter

# Create a trimmer as a runnable
trimmer = trim_messages(
    max_tokens=500,
    strategy="last",
    token_counter=model,       # Use the actual model for accurate token counting
    include_system=True,
    allow_partial=False,
    start_on="human",
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful travel assistant. Be concise."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

# Chain with trimming built in
trimmed_chain = (
    RunnablePassthrough.assign(history=lambda x: trimmer.invoke(x["history"]))
    | prompt
    | model
    | parser
)

# Wrap with message history
session_store_trimmed = {}

def get_trimmed_session(session_id: str):
    if session_id not in session_store_trimmed:
        session_store_trimmed[session_id] = InMemoryChatMessageHistory()
    return session_store_trimmed[session_id]

trimmed_chain_with_memory = RunnableWithMessageHistory(
    trimmed_chain,
    get_trimmed_session,
    input_messages_key="input",
    history_messages_key="history"
)

config = {"configurable": {"session_id": "trimmed-session"}}

# Have a conversation
turns = [
    "I want to visit Japan.",
    "What's the best season?",
    "Tell me about the food.",
    "What should I pack?",
]

for turn in turns:
    response = trimmed_chain_with_memory.invoke({"input": turn}, config=config)
    print(f"Human: {turn}")
    print(f"AI: {response}\n")

---
## YOUR TURN: Exercise A

Build a **customer support chatbot** that:
1. Has a system prompt that says it works for "AcmeAir" airlines
2. Remembers the customer's name and booking reference when they mention it
3. Can handle at least 5 turns of conversation

Test that the bot correctly references earlier context in later turns.

In [ ]:
# YOUR TURN: Build the support chatbot

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

# TODO: Create the prompt with system message and history placeholder

# TODO: Build the chain with LCEL

# TODO: Wrap with RunnableWithMessageHistory

# TODO: Run a 5-turn conversation that tests context retention
# Example turns:
# 1. "Hi, my name is Sarah and my booking is AB1234."
# 2. "I need to change my flight to Istanbul."
# 3. "What's my current booking reference?"
# 4. "Can you change it to next Friday?"
# 5. "Summarize what we've discussed."

---
## YOUR TURN: Exercise B

Build a memory-equipped chain that uses **summary-based memory** instead of storing raw messages. The idea:
1. After every 4 messages, summarize the conversation so far into a single summary message
2. Replace all old messages with the summary + the latest 2 messages
3. This keeps context small while preserving key information

Hint: You will need a separate "summarizer chain" and custom logic in your history function.

In [ ]:
# YOUR TURN: Build summary-based memory

# TODO: Create a summarizer chain that takes a list of messages and returns a summary

# TODO: Create a custom history class or function that:
#   - Stores messages normally
#   - When message count > threshold, summarize and trim

# TODO: Wire it all together and test with a long conversation

## Key Takeaways

- LLMs are **stateless** — memory must be managed externally
- **InMemoryChatMessageHistory** stores messages in a simple list
- **RunnableWithMessageHistory** automatically loads/saves history for each call
- **Session IDs** isolate different conversations
- **Message trimming** prevents context overflow — keep recent messages, drop old ones
- For production: use persistent storage (Redis, PostgreSQL) instead of in-memory

**Next:** In Exercise 4, we will build our first LangGraph — a stateful graph that gives us full control over agent logic.